<a href="https://colab.research.google.com/github/arunpradeep-g/L4-Assignments/blob/main/01-NativeLanguageTranslator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q openai gradio transformers torch torchaudio sentencepiece soundfile numpy

In [ ]:
import os
import re
import tempfile
import traceback

import numpy as np
import soundfile as sf
from google.colab import userdata
from openai import OpenAI # Added this import

api_key = userdata.get('OPENAI_API_KEY')
OPENAI_STT_MODEL = "whisper-1"
OPENAI_MT_MODEL = "gpt-4o-mini"
OPENAI_TTS_MODEL = "gpt-4o-mini-tts"
OPENAI_TTS_VOICE = "alloy"

HF_STT_MODEL = "openai/whisper-small"
HF_MT_MODEL = "facebook/nllb-200-distilled-600M"
HF_TTS_MODELS = {"en": "facebook/mms-tts-eng", "ta": "facebook/mms-tts-tam"}

NLLB_CODES = {"en": "eng_Latn", "ta": "tam_Taml"}
LANG_NAMES = {"en": "English", "ta": "Tamil"}

# Flips to True permanently once an OpenAI quota/rate limit is hit.
api_key = userdata.get('OPENAI_API_KEY')
USE_HF_FALLBACK = not bool(OpenAI(api_key=api_key))

if USE_HF_FALLBACK:
    print("OPENAI_API_KEY not found -> running on Hugging Face models only.")
else:
    print("OpenAI key detected -> OpenAI primary, Hugging Face fallback armed.")

OpenAI key detected -> OpenAI primary, Hugging Face fallback armed.


In [ ]:
TAMIL_CHARS = re.compile(r"[\u0B80-\u0BFF]")

def detect_language(text, whisper_hint=None):
    """Return 'ta' or 'en'."""
    if TAMIL_CHARS.search(text or ""):
        return "ta"
    if whisper_hint and whisper_hint.lower().startswith("ta"):
        return "ta"
    return "en"


def other(lang):
    return "en" if lang == "ta" else "ta"

In [ ]:
from openai import OpenAI, APIStatusError, RateLimitError

_openai_client = None

def openai_client():
    global _openai_client
    if _openai_client is None:
        _openai_client = OpenAI()  # reads OPENAI_API_KEY from the environment
    return _openai_client


def is_quota_error(exc):
    """True when the OpenAI token ran out of quota or hit a rate limit."""
    if isinstance(exc, RateLimitError):
        return True
    if isinstance(exc, APIStatusError) and exc.status_code in (402, 429, 529):
        return True
    text = str(exc).lower()
    return any(k in text for k in ("insufficient_quota", "rate limit", "quota", "billing"))


def openai_transcribe(audio_path):
    with open(audio_path, "rb") as fh:
        result = openai_client().audio.transcriptions.create(
            model=OPENAI_STT_MODEL,
            file=fh,
            response_format="verbose_json",
        )
    return (result.text or "").strip(), getattr(result, "language", None)


def openai_translate(text, src, tgt):
    prompt = (
        f"Translate the user's {LANG_NAMES[src]} text into {LANG_NAMES[tgt]}. "
        "Output only the translation, with no notes, transliteration or quotes."
    )
    reply = openai_client().chat.completions.create(
        model=OPENAI_MT_MODEL,
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": text},
        ],
        temperature=0.2,
    )
    return (reply.choices[0].message.content or "").strip()


def openai_speak(text, lang):
    out_path = os.path.join(tempfile.mkdtemp(), "speech.mp3")
    with openai_client().audio.speech.with_streaming_response.create(
        model=OPENAI_TTS_MODEL,
        voice=OPENAI_TTS_VOICE,
        input=text,
        instructions=f"Speak naturally in {LANG_NAMES[lang]}.",
    ) as response:
        response.stream_to_file(out_path)
    return out_path

In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, VitsModel, pipeline

_hf_cache = {}
_DEVICE = 0 if torch.cuda.is_available() else -1


def hf_transcribe(audio_path):
    if "stt" not in _hf_cache:
        print(f"Loading {HF_STT_MODEL} ...")
        _hf_cache["stt"] = pipeline(
            "automatic-speech-recognition", model=HF_STT_MODEL, device=_DEVICE
        )
    asr = _hf_cache["stt"]
    result = asr(audio_path, generate_kwargs={"task": "transcribe"}, return_timestamps=False)
    return (result["text"] or "").strip(), None


def hf_translate(text, src, tgt):
    if "mt" not in _hf_cache:
        print(f"Loading {HF_MT_MODEL} ...")
        _hf_cache["mt"] = (
            AutoTokenizer.from_pretrained(HF_MT_MODEL),
            AutoModelForSeq2SeqLM.from_pretrained(HF_MT_MODEL),
        )
    tokenizer, model = _hf_cache["mt"]
    tokenizer.src_lang = NLLB_CODES[src]
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    tokens = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids(NLLB_CODES[tgt]),
        max_new_tokens=512,
    )
    return tokenizer.batch_decode(tokens, skip_special_tokens=True)[0].strip()


def hf_speak(text, lang):
    key = f"tts_{lang}"
    if key not in _hf_cache:
        name = HF_TTS_MODELS[lang]
        print(f"Loading {name} ...")
        _hf_cache[key] = (
            AutoTokenizer.from_pretrained(name),
            VitsModel.from_pretrained(name),
        )
    tokenizer, model = _hf_cache[key]
    inputs = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        waveform = model(**inputs).waveform[0].cpu().numpy().astype(np.float32)
    out_path = os.path.join(tempfile.mkdtemp(), "speech.wav")
    sf.write(out_path, waveform, model.config.sampling_rate)
    return out_path

In [ ]:
def _switch_to_fallback(stage, exc):
    global USE_HF_FALLBACK
    USE_HF_FALLBACK = True
    print(
        f"[{stage}] OpenAI resource limit reached ({type(exc).__name__}); "
        "switching to Hugging Face models for the rest of the session."
    )


def run_stage(stage, openai_fn, hf_fn, *args):
    """Try OpenAI first; fall back to Hugging Face on quota/rate limits."""
    if not USE_HF_FALLBACK:
        try:
            return openai_fn(*args)
        except Exception as exc:
            if is_quota_error(exc):
                _switch_to_fallback(stage, exc)
            else:
                print(f"[{stage}] OpenAI error: {exc}; using Hugging Face instead.")
    return hf_fn(*args)


def translate_speech(audio_path):
    """Mic audio -> (source text, translated text, spoken translation, status)."""
    if not audio_path:
        return "", "", None, "No audio captured. Press record and speak."

    try:
        text, hint = run_stage("STT", openai_transcribe, hf_transcribe, audio_path)
        if not text:
            return "", "", None, "Nothing recognised in the recording."

        src = detect_language(text, hint)
        tgt = other(src)

        translated = run_stage("MT", openai_translate, hf_translate, text, src, tgt)
        speech_path = run_stage("TTS", openai_speak, hf_speak, translated, tgt)

        backend = "Hugging Face (fallback)" if USE_HF_FALLBACK else "OpenAI"
        status = f"{LANG_NAMES[src]} -> {LANG_NAMES[tgt]}  |  backend: {backend}"
        return text, translated, speech_path, status
    except Exception as exc:
        traceback.print_exc()
        return "", "", None, f"Error: {exc}"

In [19]:
import gradio as gr

with gr.Blocks(title="Live English <-> Tamil Translator") as demo:
    gr.Markdown(
        "## Live English <-> Tamil Translator\n"
        "Whisper STT, auto direction, spoken output. "
        "Falls back to local Hugging Face models when the OpenAI quota is exhausted. "
    )

    with gr.Row():
        mic = gr.Audio(
            sources=["microphone"],
            type="filepath",
            label="Speak (English or Tamil)",
        )
        spoken = gr.Audio(label="Translation", type="filepath", autoplay=True)

    with gr.Row():
        heard = gr.Textbox(label="Recognised speech", lines=3)
        result = gr.Textbox(label="Translation", lines=3)

    status = gr.Markdown("Ready.")

    mic.stop_recording(
        fn=translate_speech,
        inputs=mic,
        outputs=[heard, result, spoken, status],
    )

demo.launch(inline=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c36177e52d2077b8b1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [18]:
USE_HF_FALLBACK = True
print("Forced Hugging Face fallback mode.")

Forced Hugging Face fallback mode.
